# Lojistik Regresyon Değerlendirme

Bu notebook kaydedilen lojistik regresyon modelini kullanarak eğitim, doğrulama ve test bölümleri üzerinde sınama yapar; başlıca ölçütleri (accuracy, precision, recall, F1) hesaplar ve her bölüm için Türkçe görseller oluşturur.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## Model yükleme ve ölçekleme istatistikleri
- tanım: Eğitilmiş ağırlıklar ve (mean/std) testte aynı şekilde kullanılır.

In [2]:
DATA_DIR = Path('dataset')
PLOTS_DIR = DATA_DIR / 'gorseller'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train_data.csv'
VAL_PATH = DATA_DIR / 'validation_data.csv'
TEST_PATH = DATA_DIR / 'test_data.csv'
MODEL_PATH = Path('model_weights.npz')

feature_columns = ['exam_1', 'exam_2']
target_column = 'hired'

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

if not MODEL_PATH.exists():
    raise FileNotFoundError('model_weights.npz dosyası bulunamadı. Önce train.ipynb dosyasını çalıştırın.')

model_store = np.load(MODEL_PATH)
weights = model_store['weights']
bias = float(model_store['bias'][0])
feature_means = model_store['feature_means']
feature_stds = model_store['feature_stds']

def scale_features(X: np.ndarray) -> np.ndarray:
    """
    tanım:
        Eğitim istatistikleri (mean/std) ile standardizasyon uygular.
    argümanlar:
        X: Özellik matrisi.
    """
    safe_stds = np.where(feature_stds == 0.0, 1.0, feature_stds)
    return (X - feature_means) / safe_stds

def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    tanım:
        Sigmoid aktivasyon: 1/(1+e^{-z}).
    argümanlar:
        z: Lineer skor.
    """
    clipped = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-clipped))

def predict_proba(X: np.ndarray) -> np.ndarray:
    """
    tanım:
        p(y=1|x) olasılıklarını döndürür.
    argümanlar:
        X: Özellik matrisi.
    """
    linear = X @ weights + bias
    return sigmoid(linear)

def predict_labels(X: np.ndarray) -> np.ndarray:
    """
    tanım:
        İkili etiket tahmini (0/1).
    argümanlar:
        X: Özellik matrisi.
    not:
        Karar eşiği 0.5.
    """
    return (predict_proba(X) >= 0.5).astype(np.int_)

## Aktivasyon fonksiyonu (sigmoid)
- tanım: Lineer skorları olasılığa dönüştürür (0-1).
## Karar eşiği
- tanım: p ≥ 0.5 ise 1, aksi halde 0.

## Ölçütlerin tanımı
- tanım: başlıca ölçütler — accuracy (başarım), precision (kesinlik), recall (duyarlılık), F1 skoru (0/1 etikete göre).

In [3]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """
    tanım:
        İkili sınıflama ölçütlerini hesaplar.
    argümanlar:
        y_true: Gerçek etiketler (0/1)
        y_pred: Tahmin etiketleri (0/1)
    döndürür:
        accuracy, precision, recall, f1_score
    """
    y_true = y_true.astype(np.int_)
    y_pred = y_pred.astype(np.int_)

    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    tn = np.logical_and(y_pred == 0, y_true == 0).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    accuracy = (tp + tn) / max(len(y_true), 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
    }


## Parçalar üstünde değerlendirme
- tanım: train/validation/test için tahmin ve metrik hesaplama döngüsü.

In [4]:
splits = {
    'train': train_df,
    'validation': val_df,
    'test': test_df,
}

results = []

for split_name, df in splits.items():
    X_raw = df[feature_columns].to_numpy(dtype=np.float64)
    y_true = df[target_column].to_numpy(dtype=np.float64)

    X_scaled = scale_features(X_raw)
    y_pred = predict_labels(X_scaled)

    metrics = compute_metrics(y_true, y_pred)
    metrics['split'] = split_name
    results.append(metrics)

metrics_df = pd.DataFrame(results).set_index('split')
display(metrics_df.round(4))


,accuracy,precision,recall,f1_score
split,,,,
train,0.8833,0.8919,0.9167,0.9041
validation,0.9500,1.0000,0.9167,0.9565
test,0.8500,0.8462,0.9167,0.8800


## Ölçütlerin görselleştirilmesi
- tanım: Başarım/Kesinlik/Duyarlılık/F1 çubuk grafik olarak kaydedilir.

In [5]:
for split_name, row in metrics_df.iterrows():
    fig, ax = plt.subplots(figsize=(6, 4))
    metric_names = ['accuracy', 'precision', 'recall', 'f1_score']
    values = [row[m] for m in metric_names]

    bars = ax.bar(metric_names, values, color=['royalblue', 'seagreen', 'darkorange', 'firebrick'])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Skor')
    ax.set_title(f'{split_name.capitalize()} başarım ölçütleri')
    ax.grid(True, axis='y', linestyle='--', linewidth=0.5, alpha=0.6)

    fig.tight_layout()
    plot_path = PLOTS_DIR / f'{split_name}_olcutler.png'
    fig.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'{split_name} ölçüt görseli {plot_path.resolve()} dosyasına kaydedildi.')

train ölçüt görseli /home/internet/Desktop/ml/dataset/gorseller/train_olcutler.png dosyasına kaydedildi.
validation ölçüt görseli /home/internet/Desktop/ml/dataset/gorseller/validation_olcutler.png dosyasına kaydedildi.
test ölçüt görseli /home/internet/Desktop/ml/dataset/gorseller/test_olcutler.png dosyasına kaydedildi.
test ölçüt görseli /home/internet/Desktop/ml/dataset/gorseller/test_olcutler.png dosyasına kaydedildi.
